# Taller 1 - Red de Monitoreo de Calidad del Aire de Bogota.
http://rmcab.ambientebogota.gov.co/Report/stationreport

## Decripción del conjunto de datos

# 1. Instalar Librerias

In [176]:
!pip install --upgrade pip # se actualiza pip, en caso de que no lo tenga actualizado
!pip install psycopg2-binary # Psycopg es un adaptador de base de datos PostgreSQL
!pip install pandas 
!pip install openpyxl


## 1.1 Importar librerias

In [177]:
import pandas as pd
from os import walk
import os, re   #expresiones regulares

# 2. Procesar conjunto de datos (Preprocesssing-dataset)

## 2.1 Unir todos los conjuntos de datos de extensión xlsx en un arreglo 


In [178]:
# unir todos los conjuntos de datos de extensión xlsx en un arreglo 
arregloDeDataSets = []
for (dirpath, dirnames, filenames) in walk('data/raw/'):
    arregloDeDataSets.extend(filenames)
    break
arregloDeDataSets

['ca_USM_StationsReport_202231291350.xlsx',
 'ca_BOL_StationsReport_20223129119.xlsx',
 'ca_SUB_StationsReport_202231291211.xlsx',
 'ca_TUN_StationsReport_202231291241.xlsx',
 'ca_LFR_StationsReport_20223129750.xlsx',
 'ca_PTE_StationsReport_20223129110.xlsx',
 'ca_MAM_StationsReport_20223129831.xlsx',
 'ca_CBV_StationsReport_20223129333.xlsx',
 'ca_CDAR_StationsReport_20223129247.xlsx',
 'ca_SCR_StationsReport_202231291142.xlsx',
 'ca_FTB_StationsReport_20223129456.xlsx',
 'ca_JAZ_StationsReport_20223129614.xlsx',
 'ca_7MA_StationsReport_20223129920.xlsx',
 'ca_CSE_StationsReport_20223129152.xlsx',
 'ca_GYR_StationsReport_20223129531.xlsx',
 'ca_COL_StationsReport_20223129424.xlsx',
 'ca_MOV2_StationsReport_202231291026.xlsx',
 'ca_USQ_StationsReport_202231291319.xlsx',
 'ca_KEN_StationsReport_2022312979.xlsx']

## 2.2  Concatenar los datsets en uno solo 


In [ ]:
path = os.path.abspath('data/raw/') 
files = os.listdir(path)  
dfs = []
df = pd.DataFrame()
for file in files:
    if file.endswith('.xlsx'):
        dft = pd.read_excel('data/raw/'+file)
        dft['Station'] = re.search(r'_(.*?)_', str(file)).group(1)
        dfs.append(dft)

df = pd.concat(dfs, ignore_index=True)
df = df.drop(index=0).reset_index(drop=True) # eliminar la primera fila que contiene los nombres de las columnas
df.head() 

,DateTime,PM10,PM2.5,NO,NO2,NOX,SO2,CO,OZONO,Vel Viento,...,HR.1,Canal no activo,PM10 Flow,CO2,SO2 Envea,Vel Viento 10M,Dir Viento 10M,Temperatura 8M,Temperatura 20M,PM2.5 Flow
0,01-01-2021 01:00,56.6,32.7,7.504,15.962,23.493,7.24,0.44924,2.431,0.6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,01-01-2021 02:00,59.3,39.3,16.56,17.866,34.426,5.405,0.69832,1.121,0.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,01-01-2021 03:00,96.4,70.8,22.989,17.802,40.791,5.685,0.88243,1.172,0.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,01-01-2021 04:00,108.3,81,3.704,9.886,13.591,2.436,0.29549,6.565,0.7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,01-01-2021 05:00,87.7,56.1,2.098,9.272,11.371,3.039,0.16621,9.513,0.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2.3 Comprobar los valores nulos por columnas

In [181]:
#Contar cuantos valores nulos hay por cada columna
df.isnull().sum()

DateTime               18
PM10                    0
PM2.5                   0
NO                   8761
NO2                  8761
NOX                  8761
SO2                 43805
CO                      0
OZONO                8761
Vel Viento          26283
Dir Viento          26283
Temperatura         17522
HR                  52566
Presion Baro        70088
Rad Solar           61327
Station                 0
Precipitacion       61326
HR.1               157697
Canal no activo    157698
PM10 Flow          157697
CO2                148936
SO2 Envea          157697
Vel Viento 10M     157697
Dir Viento 10M     157697
Temperatura 8M     157697
Temperatura 20M    157697
PM2.5 Flow         157697
dtype: int64

## 2.4 Organizar y guardar en un nuevo df las columnas significativas

In [182]:
#Organizar y guardar en un nuevo df las columnas significativas
df = df[['PM10','PM2.5','NO','NO2','NOX','CO','OZONO','Station', 'DateTime']]
df.head()

,PM10,PM2.5,NO,NO2,NOX,CO,OZONO,Station,DateTime
0,56.6,32.7,7.504,15.962,23.493,0.44924,2.431,USM,01-01-2021 01:00
1,59.3,39.3,16.56,17.866,34.426,0.69832,1.121,USM,01-01-2021 02:00
2,96.4,70.8,22.989,17.802,40.791,0.88243,1.172,USM,01-01-2021 03:00
3,108.3,81,3.704,9.886,13.591,0.29549,6.565,USM,01-01-2021 04:00
4,87.7,56.1,2.098,9.272,11.371,0.16621,9.513,USM,01-01-2021 05:00


## 2.5 Convertir valores a numericos y sino poner NaN

In [183]:
# Convertir valores a numéricos; si no se puede convertir, poner NaN.
# Lista de columnas de contaminantes (si ya existe, la reutiliza)
cols = ['PM10','PM2.5','NO','NO2','NOX','CO','OZONO']

for c in cols:
    # normalizar comas decimales a puntos y extraer el primer número válido (si existe)
    df[c] = df[c].astype(str).str.replace(',', '.', regex=False)
    df[c] = pd.to_numeric(df[c].str.extract(r'([+-]?\d*\.?\d+)')[0], errors='coerce')

# Mostrar tipos y conteo de NaN resultante
print(df[cols].dtypes)
print("\nValores faltantes por columna después de la conversión:")
print(df[cols].isna().sum())

PM10     float64
PM2.5    float64
NO       float64
NO2      float64
NOX      float64
CO       float64
OZONO    float64
dtype: object

Valores faltantes por columna después de la conversión:
PM10     20014
PM2.5    15312
NO       27682
NO2      27680
NOX      27686
CO       31256
OZONO    32150
dtype: int64


## 2.6 Ver el tamaño del dataset 

In [184]:
# Ver el tamaño del dataset y uso de memoria
print("Tamaño (filas, columnas):", df.shape)
print(f"Nº filas: {df.shape[0]}  |  Nº columnas: {df.shape[1]}")
mem = df.memory_usage(deep=True).sum()
print(f"Uso de memoria: {mem/1024**2:.2f} MB")

Tamaño (filas, columnas): (166458, 9)
Nº filas: 166458  |  Nº columnas: 9
Uso de memoria: 27.48 MB


## 2.7 Crear una nueva columna con filtro (False o True)

In [185]:
# Convert PM2.5 to numeric first (coerce non-numeric values to NaN), then evaluate status > 12
df['PM2.5'] = pd.to_numeric(df['PM2.5'], errors='coerce')
df['Status'] = df['PM2.5'] > 12
df.Status.value_counts(dropna=False)
df.head()

,PM10,PM2.5,NO,NO2,NOX,CO,OZONO,Station,DateTime,Status
0,56.6,32.7,7.504,15.962,23.493,0.44924,2.431,USM,01-01-2021 01:00,True
1,59.3,39.3,16.560,17.866,34.426,0.69832,1.121,USM,01-01-2021 02:00,True
2,96.4,70.8,22.989,17.802,40.791,0.88243,1.172,USM,01-01-2021 03:00,True
3,108.3,81.0,3.704,9.886,13.591,0.29549,6.565,USM,01-01-2021 04:00,True
4,87.7,56.1,2.098,9.272,11.371,0.16621,9.513,USM,01-01-2021 05:00,True


## 2.8 Guardar todo el proceso en un nuevo dataset

In [ ]:
# we need another column to specify stations
df.to_csv("data/clean_data_final.csv", index = False)


# 3. Integrar nuevo dataset (Stations)

## 3.1 Cargar dataset de stations

In [152]:
# Cargar los datasets
dataset_with_missing = pd.read_csv('data/dataset_with_missing.csv')
stations = pd.read_csv('data/stations_loc.csv')

# Verificar rápidamente
print("dataset_with_missing:", dataset_with_missing.shape)
print("stations:", stations.shape)

dataset_with_missing.head()
stations.head()

dataset_with_missing: (166440, 10)
stations: (20, 10)


,estacion,Sigla,Latitud,Longitud,Altitud (m),Altura (m),Localidad,Tipo de zona,Tipo de estación,Dirección
0,guaymaral,GYR,"4°47'01.5""N","74°02'38.9""W",2580,0,Suba,Sub urbana,De fondo,Autopista Norte # 205-59
1,usaquen,USQ,"4°42'37.26""N","74°1'49.50""W",2570,10,Usaquén,Urbana,De fondo,Carrera 7B Bis # 132-11
2,suba,SUB,"4°45'40.49""N","74° 5'36.46""W",2571,6,Suba,Sub urbana,De fondo,Carrera 111 # 159A-61
3,bolivia,BOL,"4°44'08.9""N","74°07'33.2""W",2574,0,Engativá,Sub urbana,De fondo,Avenida Calle 80 # 121-98
4,las_ferias,LFR,"4°41'26.52""N","74°4'56.94""W",2552,0,Engativá,Urbana,De tráfico,Avenida Calle 80 # 69Q-50


### 3.1.1 Seleccionar algunas columnas del dataset stations

In [153]:
stations = stations[['Sigla', 'Latitud', 'Longitud', 'Localidad', 'estacion']] 


### 3.1.2 Cambiar el nombre de una columna 

In [154]:
stations = stations.rename(columns={'Sigla': 'Station', 'estacion': 'Name'}) #Columna Sigla por Station
stations.head()

,Station,Latitud,Longitud,Localidad,Name
0,GYR,"4°47'01.5""N","74°02'38.9""W",Suba,guaymaral
1,USQ,"4°42'37.26""N","74°1'49.50""W",Usaquén,usaquen
2,SUB,"4°45'40.49""N","74° 5'36.46""W",Suba,suba
3,BOL,"4°44'08.9""N","74°07'33.2""W",Engativá,bolivia
4,LFR,"4°41'26.52""N","74°4'56.94""W",Engativá,las_ferias


## 3.2 Convertir las coordenadas de texto a decimales

In [155]:
import re

def dms2dd(degrees, minutes, seconds, direction):
    dd = float(degrees) + float(minutes)/60 + float(seconds)/(60*60);
    if direction == 'S' or direction == 'W':
        dd *= -1
    return dd;

def dd2dms(deg):
    d = int(deg)
    md = abs(deg - d) * 60
    m = int(md)
    sd = (md - m) * 60
    return [d, m, sd]

def parse_dms(coor):
    parts = re.split('[^\d\w]+', coor)
    dec_coor = dms2dd(parts[0], parts[1], float(parts[2]+'.'+parts[2]), parts[4])
    return dec_coor



<>:17: SyntaxWarning: invalid escape sequence '\d'
<>:17: SyntaxWarning: invalid escape sequence '\d'
/var/folders/mk/dnyv3qdx2jv5qxqdq2h915j00000gp/T/ipykernel_42873/2119852395.py:17: SyntaxWarning: invalid escape sequence '\d'
  parts = re.split('[^\d\w]+', coor)


In [156]:
#Aplicar la funcion parse_dms a la columna Latitud y Longitud
stations['Latitud'] = stations['Latitud'].apply(parse_dms)
stations['Longitud'] = stations['Longitud'].apply(parse_dms)
stations.head()

,Station,Latitud,Longitud,Localidad,Name
0,GYR,4.783614,-74.043994,Suba,guaymaral
1,USQ,4.710381,-74.030414,Usaquén,usaquen
2,SUB,4.761222,-74.093433,Suba,suba
3,BOL,4.735578,-74.125925,Engativá,bolivia
4,LFR,4.690628,-74.082378,Engativá,las_ferias


## 3.3 Integrar el dataset df con el dataset stations

In [157]:
df = pd.merge(df, stations, on='Station', how='inner')
df.head(3)

,PM10,PM2.5,NO,NO2,NOX,CO,OZONO,Station,DateTime,Status,Latitud,Longitud,Localidad,Name
0,3.0,3.0,NaN,NaN,NaN,NaN,NaN,USM,NaN,False,4.532097,-74.116947,Usme,usame
1,56.6,32.7,7.504,15.962,23.493,0.44924,2.431,USM,01-01-2021 01:00,True,4.532097,-74.116947,Usme,usame
2,59.3,39.3,16.560,17.866,34.426,0.69832,1.121,USM,01-01-2021 02:00,True,4.532097,-74.116947,Usme,usame


## 3.4 Convertir hora (DateTime)

In [175]:
#Puede ver que en la columna 'DateTime', 
#la información sobre la fecha y la hora se dan juntas. 
#Por lo tanto, extraerá la información de tiempo.

def replace24(datetimex):
    datetimex = str(datetimex)  # Fuerza a string
    if '24:00' in datetimex:
        datetimex = datetimex.replace('24:00', '00:00')
        # Sumar un día para mantener la hora lógica (24:00 es el día siguiente)
        dt = pd.to_datetime(datetimex, dayfirst=True, errors='coerce') + pd.Timedelta(days=1)
    else:
        dt = pd.to_datetime(datetimex, dayfirst=True, errors='coerce')
    return dt
# Esta celda extraerá información de la columna 'datetime' y 
#generará columnas de meses, días o semanas y horas
df['DateTime'] = df['DateTime'].apply(replace24)
df['DateTime'] = pd.to_datetime(df['DateTime'], dayfirst=True)
df['month'] = pd.DatetimeIndex(df['DateTime']).month
df['day_week'] = pd.DatetimeIndex(df['DateTime']).weekday
df['day_month'] = pd.DatetimeIndex(df['DateTime']).day
df['hour'] = pd.DatetimeIndex(df['DateTime']).hour
df.loc[df['hour']==0,'hour'] = 24

KeyError: 'DateTime'

In [159]:
df

,PM10,PM2.5,NO,NO2,NOX,CO,OZONO,Station,DateTime,Status,Latitud,Longitud,Localidad,Name,month,day_week,day_month,hour
0,3.0,3.0,NaN,NaN,NaN,NaN,NaN,USM,NaT,False,4.532097,-74.116947,Usme,usame,NaN,NaN,NaN,NaN
1,56.6,32.7,7.504,15.962,23.493,0.44924,2.431,USM,2021-01-01 01:00:00,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,1.0
2,59.3,39.3,16.560,17.866,34.426,0.69832,1.121,USM,2021-01-01 02:00:00,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,2.0
3,96.4,70.8,22.989,17.802,40.791,0.88243,1.172,USM,2021-01-01 03:00:00,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,3.0
4,108.3,81.0,3.704,9.886,13.591,0.29549,6.565,USM,2021-01-01 04:00:00,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166454,43.1,12.0,11.771,23.697,35.468,0.98779,8.146,KEN,2021-12-31 20:00:00,False,4.625083,-74.161222,Kennedy,kennedy,12.0,4.0,31.0,20.0
166455,35.0,35.0,6.634,21.677,28.311,0.85772,8.055,KEN,2021-12-31 21:00:00,True,4.625083,-74.161222,Kennedy,kennedy,12.0,4.0,31.0,21.0
166456,43.5,43.0,2.666,19.867,22.533,0.84339,6.877,KEN,2021-12-31 22:00:00,True,4.625083,-74.161222,Kennedy,kennedy,12.0,4.0,31.0,22.0
166457,60.1,55.0,2.622,19.689,22.311,0.84038,6.533,KEN,2021-12-31 23:00:00,True,4.625083,-74.161222,Kennedy,kennedy,12.0,4.0,31.0,23.0


## 3.5 Guardar todo el proceso en un nuevo dataset

In [160]:
df.to_csv("data/dataset_with_geo_missing.csv", index = False)

# 4. Limpieza de datos

In [161]:
# Mostrar resumen de valores faltantes antes de la imputación
pollutant_cols = ['PM10','PM2.5','NO','NO2','NOX','CO','OZONO']
missing = df[pollutant_cols].isna().sum()
missing_pct = (missing / len(df)) * 100
print(pd.concat([missing, missing_pct], axis=1, keys=['missing','pct_missing']))

# Convertir a numérico (coerce a NaN si no convertible) y rellenar NaN con la media de cada columna
for col in pollutant_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col].fillna(df[col].mean(), inplace=True)

# Comprobar que las columnas de contaminantes ya no tienen valores faltantes
print("\nDespués de imputar con la media:")
print(df[pollutant_cols].isna().sum())

       missing  pct_missing
PM10     20014    12.023381
PM2.5    15312     9.198662
NO       27683    16.630522
NO2      27681    16.629320
NOX      27687    16.632925
CO       31257    18.777597
OZONO    32151    19.314666

Después de imputar con la media:
PM10     0
PM2.5    0
NO       0
NO2      0
NOX      0
CO       0
OZONO    0
dtype: int64


/var/folders/mk/dnyv3qdx2jv5qxqdq2h915j00000gp/T/ipykernel_42873/3186931200.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mean(), inplace=True)


## 4.1 Remplazar los valores NaN en cada columna con su media

In [162]:
# Rellenar las columnas de contaminantes con la media (ya se hizo PM10 arriba)
cols_to_fill = [c for c in pollutant_cols if c != 'PM10'] if 'pollutant_cols' in globals() else ['PM2.5','CO','NO','NO2','NOX','OZONO']
for c in cols_to_fill:
    df[c].fillna(df[c].mean(), inplace=True)

# Verificación rápida
print(df[cols_to_fill].isnull().sum())

PM2.5    0
NO       0
NO2      0
NOX      0
CO       0
OZONO    0
dtype: int64


/var/folders/mk/dnyv3qdx2jv5qxqdq2h915j00000gp/T/ipykernel_42873/307701874.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[c].fillna(df[c].mean(), inplace=True)


In [163]:
# compruebe --- ya no hay valores perdidos en el dataset

df.isnull().sum()

PM10          0
PM2.5         0
NO            0
NO2           0
NOX           0
CO            0
OZONO         0
Station       0
DateTime     19
Status        0
Latitud       0
Longitud      0
Localidad     0
Name          0
month        19
day_week     19
day_month    19
hour         19
dtype: int64

## 4.2 Guardar todo el proceso en un nuevo dataset

In [164]:
df.to_csv("data/dataset_final_clean_mean.csv", index = False)

# 5. Creando el esquema de la bodega de datos 

In [165]:
df = pd.read_csv('data/dataset_final_clean_mean.csv')
df.head(10)

,PM10,PM2.5,NO,NO2,NOX,CO,OZONO,Station,DateTime,Status,Latitud,Longitud,Localidad,Name,month,day_week,day_month,hour
0,3.0,3.0,18.616861,14.979247,33.5903,0.691262,11.782188,USM,NaN,False,4.532097,-74.116947,Usme,usame,NaN,NaN,NaN,NaN
1,56.6,32.7,7.504000,15.962000,23.4930,0.449240,2.431000,USM,2021-01-01 01:00:00,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,1.0
2,59.3,39.3,16.560000,17.866000,34.4260,0.698320,1.121000,USM,2021-01-01 02:00:00,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,2.0
3,96.4,70.8,22.989000,17.802000,40.7910,0.882430,1.172000,USM,2021-01-01 03:00:00,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,3.0
4,108.3,81.0,3.704000,9.886000,13.5910,0.295490,6.565000,USM,2021-01-01 04:00:00,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,4.0
5,87.7,56.1,2.098000,9.272000,11.3710,0.166210,9.513000,USM,2021-01-01 05:00:00,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,5.0
6,74.4,38.6,2.249000,11.064000,13.3130,0.225340,5.466000,USM,2021-01-01 06:00:00,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,6.0
7,60.4,27.2,11.490000,14.675000,26.1650,0.413570,2.467000,USM,2021-01-01 07:00:00,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,7.0
8,51.9,29.5,5.864000,7.348000,13.2120,0.207840,10.800000,USM,2021-01-01 08:00:00,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,8.0
9,38.1,21.6,3.506000,4.266000,7.7730,0.691262,17.161000,USM,2021-01-01 09:00:00,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,9.0


## 5.1 Se crea un nuevo DataFrame con los datos de los polutantes

In [166]:
df_polutante=pd.DataFrame(df, columns=["PM10", "PM2.5", "NO", "NO2", "NOX", "CO", "OZONO"]) # se crea un nuevo DataFrame unicamente con los polutantes
df['id_polutante'] = df.index+1 # se crea una nueva columna que identificará a cada polutante
df_polutante

,PM10,PM2.5,NO,NO2,NOX,CO,OZONO
0,3.0,3.0,18.616861,14.979247,33.5903,0.691262,11.782188
1,56.6,32.7,7.504000,15.962000,23.4930,0.449240,2.431000
2,59.3,39.3,16.560000,17.866000,34.4260,0.698320,1.121000
3,96.4,70.8,22.989000,17.802000,40.7910,0.882430,1.172000
4,108.3,81.0,3.704000,9.886000,13.5910,0.295490,6.565000
...,...,...,...,...,...,...,...
166454,43.1,12.0,11.771000,23.697000,35.4680,0.987790,8.146000
166455,35.0,35.0,6.634000,21.677000,28.3110,0.857720,8.055000
166456,43.5,43.0,2.666000,19.867000,22.5330,0.843390,6.877000
166457,60.1,55.0,2.622000,19.689000,22.3110,0.840380,6.533000


## 5.2 Se crea un nuevo dataset con los datos de las estaciones y se eliminan las filas repetidas

In [167]:
df

,PM10,PM2.5,NO,NO2,NOX,CO,OZONO,Station,DateTime,Status,Latitud,Longitud,Localidad,Name,month,day_week,day_month,hour,id_polutante
0,3.0,3.0,18.616861,14.979247,33.5903,0.691262,11.782188,USM,NaN,False,4.532097,-74.116947,Usme,usame,NaN,NaN,NaN,NaN,1
1,56.6,32.7,7.504000,15.962000,23.4930,0.449240,2.431000,USM,2021-01-01 01:00:00,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,1.0,2
2,59.3,39.3,16.560000,17.866000,34.4260,0.698320,1.121000,USM,2021-01-01 02:00:00,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,2.0,3
3,96.4,70.8,22.989000,17.802000,40.7910,0.882430,1.172000,USM,2021-01-01 03:00:00,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,3.0,4
4,108.3,81.0,3.704000,9.886000,13.5910,0.295490,6.565000,USM,2021-01-01 04:00:00,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,4.0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166454,43.1,12.0,11.771000,23.697000,35.4680,0.987790,8.146000,KEN,2021-12-31 20:00:00,False,4.625083,-74.161222,Kennedy,kennedy,12.0,4.0,31.0,20.0,166455
166455,35.0,35.0,6.634000,21.677000,28.3110,0.857720,8.055000,KEN,2021-12-31 21:00:00,True,4.625083,-74.161222,Kennedy,kennedy,12.0,4.0,31.0,21.0,166456
166456,43.5,43.0,2.666000,19.867000,22.5330,0.843390,6.877000,KEN,2021-12-31 22:00:00,True,4.625083,-74.161222,Kennedy,kennedy,12.0,4.0,31.0,22.0,166457
166457,60.1,55.0,2.622000,19.689000,22.3110,0.840380,6.533000,KEN,2021-12-31 23:00:00,True,4.625083,-74.161222,Kennedy,kennedy,12.0,4.0,31.0,23.0,166458


In [168]:
df_estacion=pd.DataFrame(df, columns=["Name", "Station", "Localidad", "Latitud", "Longitud"]) # se crea un nuevo DataFrame unicamente con los datos de las estaciones
df_estacion = df_estacion.drop_duplicates() # se eliminan las filas repetidas
df_estacion = df_estacion.rename(columns={'Station': 'Sigla'}) # se cambia el nombre a la columna Station por Sigla
df_estacion

,Name,Sigla,Localidad,Latitud,Longitud
0,usame,USM,Usme,4.532097,-74.116947
8761,bolivia,BOL,Engativá,4.735578,-74.125925
17522,suba,SUB,Suba,4.761222,-74.093433
26283,tunal,TUN,Tunjuelito,4.576206,-74.130975
35044,las_ferias,LFR,Engativá,4.690628,-74.082378
43805,pueste_aranda,PTE,Puente Aranda,4.631817,-74.117278
52566,minambiente,MAM,Santa Fe,4.625364,-74.066972
61327,ciudad_bolivar,CBV,Ciudad Bolívar,4.577889,-74.166272
70088,centro_alto_rendimiento,CDAR,Engativá,4.658417,-74.083944
78849,san_cristobal,SCR,San Cristóbal,4.572558,-74.083639


In [169]:
# se remplaza la columna sigla, dejando unicamente los datos unicos y luego se le cambia el nombre
# a la misma por id_estacion
count = 1
for index, row in df_estacion.iterrows():
    df = df.replace({row["Sigla"]: count})
    count += 1
df = df.rename(columns={'Station': 'id_estacion'})
df

/var/folders/mk/dnyv3qdx2jv5qxqdq2h915j00000gp/T/ipykernel_42873/461078715.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace({row["Sigla"]: count})


,PM10,PM2.5,NO,NO2,NOX,CO,OZONO,id_estacion,DateTime,Status,Latitud,Longitud,Localidad,Name,month,day_week,day_month,hour,id_polutante
0,3.0,3.0,18.616861,14.979247,33.5903,0.691262,11.782188,1,NaN,False,4.532097,-74.116947,Usme,usame,NaN,NaN,NaN,NaN,1
1,56.6,32.7,7.504000,15.962000,23.4930,0.449240,2.431000,1,2021-01-01 01:00:00,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,1.0,2
2,59.3,39.3,16.560000,17.866000,34.4260,0.698320,1.121000,1,2021-01-01 02:00:00,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,2.0,3
3,96.4,70.8,22.989000,17.802000,40.7910,0.882430,1.172000,1,2021-01-01 03:00:00,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,3.0,4
4,108.3,81.0,3.704000,9.886000,13.5910,0.295490,6.565000,1,2021-01-01 04:00:00,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,4.0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166454,43.1,12.0,11.771000,23.697000,35.4680,0.987790,8.146000,19,2021-12-31 20:00:00,False,4.625083,-74.161222,Kennedy,kennedy,12.0,4.0,31.0,20.0,166455
166455,35.0,35.0,6.634000,21.677000,28.3110,0.857720,8.055000,19,2021-12-31 21:00:00,True,4.625083,-74.161222,Kennedy,kennedy,12.0,4.0,31.0,21.0,166456
166456,43.5,43.0,2.666000,19.867000,22.5330,0.843390,6.877000,19,2021-12-31 22:00:00,True,4.625083,-74.161222,Kennedy,kennedy,12.0,4.0,31.0,22.0,166457
166457,60.1,55.0,2.622000,19.689000,22.3110,0.840380,6.533000,19,2021-12-31 23:00:00,True,4.625083,-74.161222,Kennedy,kennedy,12.0,4.0,31.0,23.0,166458


## 5.3 Se crea un nuevo DataFrame con los datos de las fechas.

In [170]:
# se crea un nuevo DataFrame y se eliminan los valores repetidos
df_fecha=pd.DataFrame(df, columns=["DateTime"]) # se crea un nuevo DataFrame con las fechas y horas 
#df_fecha["DateTime"].value_counts() # !TODO Borrar?
df_fecha = df_fecha.drop_duplicates() # se eliminan las filas repetidas
df_fecha

,DateTime
0,NaN
1,2021-01-01 01:00:00
2,2021-01-01 02:00:00
3,2021-01-01 03:00:00
4,2021-01-01 04:00:00
...,...
8756,2021-12-31 20:00:00
8757,2021-12-31 21:00:00
8758,2021-12-31 22:00:00
8759,2021-12-31 23:00:00


In [171]:
import re # libreria para el uso de expresiones regulares
from pandas.tseries.holiday import Holiday, AbstractHolidayCalendar
from datetime import datetime

# definimos los días festivo para el año en que se tomaron los datos
class EsBusinessCalendar(AbstractHolidayCalendar):
   rules = [ # se definen los festivos, teniendo en cuenta los horarios festivos para el año de 2021 en Colombia
     Holiday('Año Nuevo', month=1, day=1),
     Holiday('Día de los Reyes Magos', month=1, day=11),
     Holiday('Día de San José', month=3, day=22),
     Holiday('Jueves Santo', month=4, day=1),
     Holiday('Viernes Santo', month=4, day=2),
     Holiday('Día del Trabajador', month=5, day=1),
     Holiday('Día de la Ascensión', month=5, day=17),
     Holiday('Corpus Christi', month=6, day=7),
     Holiday('Sagrado Corazón', month=6, day=14),
     Holiday('San Pedro y San Pablo ', month=7, day=5),
     Holiday('Día de la Independencia', month=7, day=20),
     Holiday('Batalla de Boyacá', month=8, day=7),
     Holiday('Asunción de la Virgen', month=8, day=16),
     Holiday('Celebración del Día de la Raza', month=10, day=18),
     Holiday('Día de todos los Santos', month=11, day=1),
     Holiday('Independencia de Cartagena', month=11, day=15),
     Holiday('Inmaculada Concepción', month=12, day=8),    
     Holiday('Navidad', month=12, day=25)
   ]

calendar_festivos = EsBusinessCalendar() # se instancia el objeto
calendar_festivos = calendar_festivos.holidays(start='2021-01-01', end='2021-12-31') # se define el rango de tiempo en que se tendran en cuenta los festivos (anho 2021)

dias = []
meses = []
anhos = []
horas = []
fin_semana = []
festivo = []

# Recorremos filas manejando valores NaN y distintos tipos (str, Timestamp, datetime)
for index, row in df_fecha.iterrows(): # se recorre cada fila del DataFrame
    dt = row['DateTime']
    # Si el valor es NaN (float) o None, agregamos valores nulos coherentes
    if pd.isna(dt):
        dias.append(None)
        meses.append(None)
        anhos.append(None)
        horas.append(None)
        fin_semana.append(False)
        festivo.append(False)
        continue

    # Convertir a Timestamp de forma robusta (intenta dayfirst=False y luego dayfirst=True)
    parsed = pd.to_datetime(dt, errors='coerce', dayfirst=False)
    if pd.isna(parsed):
        parsed = pd.to_datetime(dt, errors='coerce', dayfirst=True)

    if pd.isna(parsed):
        # si no se pudo parsear, marcar como nulo
        dias.append(None)
        meses.append(None)
        anhos.append(None)
        horas.append(None)
        fin_semana.append(False)
        festivo.append(False)
        continue

    # Extraer componentes
    dias.append(parsed.day)
    meses.append(parsed.month)
    anhos.append(parsed.year)
    # usar 24 para la hora 0 si así lo desea; aquí guardamos la hora como entero (0-23)
    hour_val = parsed.hour
    # según la lógica previa, quizá se quería reemplazar 0 por 24; si desea eso, descomente:
    # if hour_val == 0:
    #     hour_val = 24
    horas.append(hour_val)

    # fin de semana: True si sábado(5) o domingo(6)
    fin_semana.append(parsed.weekday() >= 5)

    # festivo: comparar la fecha normalizada con la lista de festivos
    festivo.append(parsed.normalize() in calendar_festivos)

dict_fechas = { 'dia': dias, 'mes': meses, 'anho': anhos, 'hora': horas, 'fin_semana': fin_semana, 'festivo': festivo}
df_fechas = pd.DataFrame(data=dict_fechas)
df_fechas
class EsBusinessCalendar(AbstractHolidayCalendar):
   rules = [ # se definen los festivos, teniendo en cuenta los horarios festivos para el año de 2021 en Colombia
     Holiday('Año Nuevo', month=1, day=1),
     Holiday('Día de los Reyes Magos', month=1, day=11),
     Holiday('Día de San José', month=3, day=22),
     Holiday('Jueves Santo', month=4, day=1),
     Holiday('Viernes Santo', month=4, day=2),
     Holiday('Día del Trabajador', month=5, day=1),
     Holiday('Día de la Ascensión', month=5, day=17),
     Holiday('Corpus Christi', month=6, day=7),
     Holiday('Sagrado Corazón', month=6, day=14),
     Holiday('San Pedro y San Pablo ', month=7, day=5),
     Holiday('Día de la Independencia', month=7, day=20),
     Holiday('Batalla de Boyacá', month=8, day=7),
     Holiday('Asunción de la Virgen', month=8, day=16),
     Holiday('Celebración del Día de la Raza', month=10, day=18),
     Holiday('Día de todos los Santos', month=11, day=1),
     Holiday('Independencia de Cartagena', month=11, day=15),
     Holiday('Inmaculada Concepción', month=12, day=8),    
     Holiday('Navidad', month=12, day=25)
   ]

calendar_festivos = EsBusinessCalendar() # se instancia el objeto
calendar_festivos = calendar_festivos.holidays(start='2021-01-01', end='2021-12-31') # se define el rango de tiempo en que se tendran en cuenta los festivos (anho 2021)

dias = []
meses = []
anhos = []
horas = []
fin_semana = []
festivo = []

# Recorremos filas manejando valores NaN y distintos tipos (str, Timestamp, datetime)
for index, row in df_fecha.iterrows(): # se recorre cada fila del DataFrame
    dt = row['DateTime']
    # Si el valor es NaN (float) o None, agregamos valores nulos coherentes
    if pd.isna(dt):
        dias.append(None)
        meses.append(None)
        anhos.append(None)
        horas.append(None)
        fin_semana.append(False)
        festivo.append(False)
        continue

    # Convertir a Timestamp de forma robusta (intenta dayfirst=False y luego dayfirst=True)
    parsed = pd.to_datetime(dt, errors='coerce', dayfirst=False)
    if pd.isna(parsed):
        parsed = pd.to_datetime(dt, errors='coerce', dayfirst=True)

    if pd.isna(parsed):
        # si no se pudo parsear, marcar como nulo
        dias.append(None)
        meses.append(None)
        anhos.append(None)
        horas.append(None)
        fin_semana.append(False)
        festivo.append(False)
        continue

    # Extraer componentes
    dias.append(parsed.day)
    meses.append(parsed.month)
    anhos.append(parsed.year)
    # usar 24 para la hora 0 si así lo desea; aquí guardamos la hora como entero (0-23)
    hour_val = parsed.hour
    # según la lógica previa, quizá se quería reemplazar 0 por 24; si desea eso, descomente:
    # if hour_val == 0:
    #     hour_val = 24
    horas.append(hour_val)

    # fin de semana: True si sábado(5) o domingo(6)
    fin_semana.append(parsed.weekday() >= 5)

    # festivo: comparar la fecha normalizada con la lista de festivos
    festivo.append(parsed.normalize() in calendar_festivos)

dict_fechas = { 'dia': dias, 'mes': meses, 'anho': anhos, 'hora': horas, 'fin_semana': fin_semana, 'festivo': festivo}
df_fechas = pd.DataFrame(data=dict_fechas)
df_fechas


,dia,mes,anho,hora,fin_semana,festivo
0,NaN,NaN,NaN,NaN,False,False
1,1.0,1.0,2021.0,1.0,False,True
2,1.0,1.0,2021.0,2.0,False,True
3,1.0,1.0,2021.0,3.0,False,True
4,1.0,1.0,2021.0,4.0,False,True
...,...,...,...,...,...,...
8756,31.0,12.0,2021.0,20.0,False,False
8757,31.0,12.0,2021.0,21.0,False,False
8758,31.0,12.0,2021.0,22.0,False,False
8759,31.0,12.0,2021.0,23.0,False,False


## 5.4 Creación de la columna id_tiempo para identificar cada una de las fechas en el DF principal

In [172]:
count = 1
# se recorre cada fecha de df_fecha
for index, row in df_fecha.iterrows(): 
# por cada fila de df_fecha se crea un nuevo identificador en el DataFrame principal
    df.loc[df['DateTime'] == row["DateTime"], 'DateTime'] = count     
    count += 1
df = df.rename(columns={'DateTime': 'id_tiempo'})
df

,PM10,PM2.5,NO,NO2,NOX,CO,OZONO,id_estacion,id_tiempo,Status,Latitud,Longitud,Localidad,Name,month,day_week,day_month,hour,id_polutante
0,3.0,3.0,18.616861,14.979247,33.5903,0.691262,11.782188,1,NaN,False,4.532097,-74.116947,Usme,usame,NaN,NaN,NaN,NaN,1
1,56.6,32.7,7.504000,15.962000,23.4930,0.449240,2.431000,1,2,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,1.0,2
2,59.3,39.3,16.560000,17.866000,34.4260,0.698320,1.121000,1,3,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,2.0,3
3,96.4,70.8,22.989000,17.802000,40.7910,0.882430,1.172000,1,4,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,3.0,4
4,108.3,81.0,3.704000,9.886000,13.5910,0.295490,6.565000,1,5,True,4.532097,-74.116947,Usme,usame,1.0,4.0,1.0,4.0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166454,43.1,12.0,11.771000,23.697000,35.4680,0.987790,8.146000,19,8757,False,4.625083,-74.161222,Kennedy,kennedy,12.0,4.0,31.0,20.0,166455
166455,35.0,35.0,6.634000,21.677000,28.3110,0.857720,8.055000,19,8758,True,4.625083,-74.161222,Kennedy,kennedy,12.0,4.0,31.0,21.0,166456
166456,43.5,43.0,2.666000,19.867000,22.5330,0.843390,6.877000,19,8759,True,4.625083,-74.161222,Kennedy,kennedy,12.0,4.0,31.0,22.0,166457
166457,60.1,55.0,2.622000,19.689000,22.3110,0.840380,6.533000,19,8760,True,4.625083,-74.161222,Kennedy,kennedy,12.0,4.0,31.0,23.0,166458


In [173]:
# finalmente se crea un DataFrame, el cual se relaciona con los DataFrame creados anteriormente por medio de sus identificadores 
df_fact_medidad=pd.DataFrame(df, columns=["id_estacion", "id_tiempo", "id_polutante"])
df_fact_medidad

,id_estacion,id_tiempo,id_polutante
0,1,NaN,1
1,1,2,2
2,1,3,3
3,1,4,4
4,1,5,5
...,...,...,...
166454,19,8757,166455
166455,19,8758,166456
166456,19,8759,166457
166457,19,8760,166458


## 5.6Creación de las tablas en Postgres

In [ ]:
from importlib import reload
import conexion
reload(conexion)
new_model = conexion.new_model

# Evitar ValueError al convertir columnas con NaN a enteros: usar Int64 (nullable integer)
df_fact_medidad['id_estacion'] = pd.to_numeric(df_fact_medidad['id_estacion'], errors='coerce').astype('Int64')
df_fact_medidad['id_polutante'] = pd.to_numeric(df_fact_medidad['id_polutante'], errors='coerce').astype('Int64')
df_fact_medidad['id_tiempo'] = pd.to_numeric(df_fact_medidad['id_tiempo'], errors='coerce').astype('Int64')

# Llamadas a new_model envueltas en try/except para capturar problemas de conexión y dar sugerencias útiles
try:
	new_model(df_estacion, "dim_estacion")
	new_model(df_polutante, "dim_polutante")
	new_model(df_fechas, "dim_tiempo")
	new_model(df_fact_medidad, "fact_medidad")
except Exception as e:
	print("Error al crear las tablas con new_model:", type(e).__name__, e)
	import traceback
	traceback.print_exc()